In [1]:
import sys
sys.path.append(r'C:\Users\BBBS-AI-01\d\rl\unity_proj\unity-hide-and-seek')

In [2]:
# show sys.path entries
for i, p in enumerate(sys.path):
    print(f"{i:2d}: {p}")

 0: c:\Users\BBBS-AI-01\.conda\envs\py10\python310.zip
 1: c:\Users\BBBS-AI-01\.conda\envs\py10\DLLs
 2: c:\Users\BBBS-AI-01\.conda\envs\py10\lib
 3: c:\Users\BBBS-AI-01\.conda\envs\py10
 4: 
 5: c:\Users\BBBS-AI-01\.conda\envs\py10\lib\site-packages
 6: c:\Users\BBBS-AI-01\.conda\envs\py10\lib\site-packages\win32
 7: c:\Users\BBBS-AI-01\.conda\envs\py10\lib\site-packages\win32\lib
 8: c:\Users\BBBS-AI-01\.conda\envs\py10\lib\site-packages\Pythonwin
 9: C:\Users\BBBS-AI-01\d\rl\unity_proj\unity-hide-and-seek


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import glob
import os

# Helper to read CSV and filter only rows with exact columns
def read_csv_filtered(file_path, expected_cols):
    valid_rows = []
    malformed_times = set()
    with open(file_path, "r") as f:
        header = f.readline().strip().split(",")
        for line in f:
            parts = line.strip().split(",")
            if len(parts) == expected_cols:
                valid_rows.append(parts)
            else:
                # Capture timestamp if malformed
                if len(parts) > 0:
                    try:
                        malformed_times.add(float(parts[0]))
                    except ValueError:
                        pass
    df = pd.DataFrame(valid_rows, columns=header[:expected_cols])
    for col in df.columns:
        if col != "block_id":  # keep block_id as string
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df, malformed_times

# Load blocks (7 columns)
# build absolute path using the notebook variable `p`
blocks_path = os.path.join(p, "trajectory_logs_/run41/episode_5/blocks.csv")
print("Reading:", blocks_path, "exists:", os.path.exists(blocks_path))

blocks_df, malformed_blocks = read_csv_filtered(blocks_path, 7)

# Load agents (5 columns)
agent_files = glob.glob("trajectory_logs_/run41/episode_5/agent_*.csv")
agents = {}
malformed_agents = set()
for file in agent_files:
    df, malformed_times = read_csv_filtered(file, 5)
    name = file.split("/")[-1].replace(".csv", "")
    agents[name] = df
    malformed_agents.update(malformed_times)

# Combine all malformed timestamps
all_malformed = malformed_blocks.union(malformed_agents)
print("Malformed timestamps to ignore:", sorted(all_malformed))

# Drop malformed timestamps from data
blocks_df = blocks_df[~blocks_df['time'].isin(all_malformed)]
for name in agents:
    agents[name] = agents[name][~agents[name]['time'].isin(all_malformed)]

# Ensure no remaining NaNs
blocks_df.dropna(inplace=True)
for name in agents:
    agents[name].dropna(inplace=True)

# Unique timestamps (valid only)
timestamps = sorted(blocks_df['time'].unique())

# Create frames
frames = []
for t in timestamps:
    blocks_t = blocks_df[blocks_df['time'] == t]
    block_trace = go.Scatter3d(
        x=blocks_t['x'],
        y=blocks_t['z'],  # Z as vertical
        z=[1]*len(blocks_t),  # horizontal plane
        mode='markers',
        marker=dict(size=5, color='brown'),
        name='Blocks'
    )

    agent_traces = []
    for name, df in agents.items():
        df_t = df[df['time'] == t]
        agent_traces.append(
            go.Scatter3d(
                x=df_t['x'],
                y=df_t['z'],
                z=[1]*len(df_t),
                mode='markers',
                marker=dict(size=5, color='blue' if "Hider" in name else 'red'),
                name=name
            )
        )

    frames.append(go.Frame(data=[block_trace] + agent_traces, name=str(t)))

# Initial frame (first valid timestamp)
init_t = timestamps[0]
init_blocks = blocks_df[blocks_df['time'] == init_t]
init_block_trace = go.Scatter3d(
    x=init_blocks['x'],
    y=init_blocks['z'],
    z=[1]*len(init_blocks),
    mode='markers',
    marker=dict(size=5, color='brown'),
    name='Blocks'
)

init_agent_traces = []
for name, df in agents.items():
    df_t = df[df['time'] == init_t]
    init_agent_traces.append(
        go.Scatter3d(
            x=df_t['x'],
            y=df_t['z'],
            z=[1]*len(df_t),
            mode='markers',
            marker=dict(size=5, color='blue' if "Hider" in name else 'red'),
            name=name
        )
    )

In [14]:
# Compare rows per timestamp between blocks and all agents
# Uses existing vars: blocks_df, agents

# Build union of all timestamps present
all_times = sorted(
    set(blocks_df['time'].unique()).union(*[set(df['time'].unique()) for df in agents.values()])
)

# expected counts
expected_block_count = blocks_df['block_id'].nunique()
expected_agent_count = 1  # usually one row per agent per timestamp

# Build a comparison table: rows = time, columns = blocks + each agent
rows = []
for t in all_times:
    row = {"time": t}
    row["blocks_count"] = int(len(blocks_df[blocks_df['time'] == t]))
    for name, df in agents.items():
        row[name] = int(len(df[df['time'] == t]))
    rows.append(row)

comp_df = pd.DataFrame(rows).set_index('time').sort_index()

# Add helper columns
comp_df['agents_present'] = (comp_df[list(agents.keys())] > 0).sum(axis=1)
comp_df['any_agent_missing'] = (comp_df[list(agents.keys())] != expected_agent_count).any(axis=1)
comp_df['blocks_mismatch'] = comp_df['blocks_count'] != expected_block_count

# Summary
print("Expected blocks per timestamp:", expected_block_count)
print("Expected rows per agent per timestamp:", expected_agent_count)
print("\nTotals:")
print("  timestamps checked:", len(comp_df))
print("  timestamps with any agent missing or extra rows:", comp_df['any_agent_missing'].sum())
print("  timestamps with blocks count mismatch:", comp_df['blocks_mismatch'].sum())
print("  timestamps where not all agents present:", (comp_df['agents_present'] != len(agents)).sum())

# Show first problematic rows
bad_mask = comp_df['any_agent_missing'] | comp_df['blocks_mismatch']
if bad_mask.any():
    print("\nFirst 20 timestamps with discrepancies:")
    display(comp_df[bad_mask].head(20))
else:
    print("\nNo discrepancies found. All agent rows and block counts match expectations.")

# Optional: export comparison for further inspection
comp_df.to_csv("comparison_rows_per_time.csv")
print("\nSaved comparison_rows_per_time.csv")

Expected blocks per timestamp: 8
Expected rows per agent per timestamp: 1

Totals:
  timestamps checked: 498
  timestamps with any agent missing or extra rows: 0
  timestamps with blocks count mismatch: 498
  timestamps where not all agents present: 0

First 20 timestamps with discrepancies:


,blocks_count,agents_present,any_agent_missing,blocks_mismatch
time,,,,
30.04,3,0.0,False,True
30.06,4,0.0,False,True
30.08,4,0.0,False,True
30.10,4,0.0,False,True
30.12,4,0.0,False,True
30.14,4,0.0,False,True
30.16,4,0.0,False,True
30.18,4,0.0,False,True
30.20,4,0.0,False,True



Saved comparison_rows_per_time.csv


In [15]:
print("Unique block_ids:", blocks_df['block_id'].unique())
print("Value counts:", blocks_df['block_id'].value_counts())

Unique block_ids: ['Block1' 'Block2' 'Block3' 'Block0' 'Block4' 'Block5' 'Block6' 'Block7']
Value counts: block_id
Block1    831
Block2    831
Block3    831
Block0    827
Block4    321
Block5    215
Block6    134
Block7     63
Name: count, dtype: int64


In [13]:
# Basic summary: counts, time range, per-agent row counts
print("Blocks rows:", len(blocks_df))
print("Time span:", blocks_df['time'].min(), "to", blocks_df['time'].max())
print("Unique blocks:", blocks_df['block_id'].nunique())
print("\nPer-agent counts:")
for name, df in agents.items():
    print(name, len(df), "rows, time range:", df['time'].min(), "to", df['time'].max())


Blocks rows: 4053
Time span: 30.04 to 39.98
Unique blocks: 8

Per-agent counts:


In [16]:
print("Last 10 timestamps in comp_df:")
display(comp_df.tail(10))

Last 10 timestamps in comp_df:


,blocks_count,agents_present,any_agent_missing,blocks_mismatch
time,,,,
39.80,34,0.0,False,True
39.82,34,0.0,False,True
39.84,34,0.0,False,True
39.86,34,0.0,False,True
39.88,34,0.0,False,True
39.90,34,0.0,False,True
39.92,34,0.0,False,True
39.94,34,0.0,False,True
39.96,34,0.0,False,True


In [17]:
last_time = blocks_df['time'].max()
print(f"At last time {last_time}, blocks_count: {len(blocks_df[blocks_df['time'] == last_time])}")
print("Blocks present:", blocks_df[blocks_df['time'] == last_time]['block_id'].tolist())

At last time 39.98, blocks_count: 34
Blocks present: ['Block0', 'Block1', 'Block2', 'Block3', 'Block0', 'Block1', 'Block2', 'Block3', 'Block0', 'Block1', 'Block2', 'Block3', 'Block4', 'Block0', 'Block1', 'Block2', 'Block3', 'Block4', 'Block5', 'Block0', 'Block1', 'Block2', 'Block3', 'Block4', 'Block5', 'Block6', 'Block0', 'Block1', 'Block2', 'Block3', 'Block4', 'Block5', 'Block6', 'Block7']


In [18]:
# Show blocks at a specific timestamp
specific_time = 39.98  # Change this to any timestamp you want
blocks_at_time = blocks_df[blocks_df['time'] == specific_time]
print(f"Blocks at time {specific_time}:")
display(blocks_at_time)

Blocks at time 39.98:


,time,block_id,x,y,z,isLocked,isHeld
1991,39.98,Block0,10.6175,1.0,10.9324,0,0
1992,39.98,Block1,-0.6202,1.0,-0.1550,1,0
1993,39.98,Block2,5.9324,1.0,-4.6654,0,0
1994,39.98,Block3,-5.5285,1.0,-2.9677,0,1
2039,39.98,Block0,-5.4833,-9.0,-11.0315,0,0
2040,39.98,Block1,-2.1711,-9.0,-8.3351,0,0
2041,39.98,Block2,-3.3079,-9.0,9.8825,1,0
2042,39.98,Block3,9.3010,-9.0,3.2632,0,0
2568,39.98,Block0,4.6605,-39.0,4.5902,1,0
2569,39.98,Block1,-6.9967,-39.0,-7.8040,0,0


In [19]:
# Show agents at a specific timestamp
specific_time = 39.98  # Change this to any timestamp you want
print(f"Agents at time {specific_time}:")
for name, df in agents.items():
    agent_at_time = df[df['time'] == specific_time]
    print(f"\n{name}: {len(agent_at_time)} rows")
    if not agent_at_time.empty:
        display(agent_at_time)

Agents at time 39.98:
